# योजना सारथी · Yojana Sarathi

## The scheme is not the hard part. The interview is.

An offline tool that works out which welfare schemes a household can actually claim, by
conducting the interview nobody currently has time to conduct.

*Build with Gemma: TFUG Prayagraj \[AI Prayagraj\]* · **Code:** https://github.com/yunyiliu/kaggle-gemma-yojana-sarathi

---

Uttar Pradesh publishes its welfare schemes. The eligibility rules are not secret and they
are not subtle: an age, an income ceiling, a ration-card category, whether the roof is
thatch or concrete. Anyone who sat down with the circulars and a household for an hour
could work out exactly what that household is owed.

Nobody has the hour.

A frontline worker in a village has a queue behind her. The household in front of her has
a harvest to get in, or a child to collect. The fact set that decides the eight schemes in
this notebook is **twenty-one fields long** — and asked as a form, in form order, most of
those questions are dead weight for any particular household. So the interview does not
happen, and the entitlement stays theoretical.

That is the actual failure. It is not a knowledge problem. It is a **triage problem under
a time budget**, and it is the one this project attacks.

## The claim, stated up front so it can be checked

The engine picks each question by scoring every unknown field against the money it would
unlock — the benefit currently sitting in *undecided* that knowing this one fact would
resolve, divided across whatever else each scheme is still waiting on.

That is a claim about an algorithm, so it gets measured rather than asserted: eight
complete households, four question-ordering strategies, **no model in the loop at all**.
The result is the table in section 4, and the short version is:

> After **one question**, the shipped strategy has established just over **half** of what
> the average household is owed. A paper form asking the same fields in a fixed order has
> established **nothing** until question four.

Which matters because interviews get abandoned — someone is called away, a queue moves, a
child starts crying. What you have secured when the conversation stops *is* the product.

## 1 · Environment

An Ollama server on `localhost`. No API key appears anywhere in this notebook and no
request leaves the machine.

That is not a preference. The details this tool exists to collect — a bereavement, an
income, a caste certificate — belong to the household. Connectivity is thinnest exactly
where the need is greatest. And a per-query bill does not survive being multiplied by a
state's frontline workforce.

Two dependencies the Ollama installer needs that Kaggle's image does not ship: `zstd`,
without which extraction fails outright, and `pciutils`, without which the installer
cannot see the GPU and quietly installs the CPU-only build.

In [ ]:
import json, os, subprocess, sys, textwrap, time
t0 = time.time()

!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y zstd pciutils > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh 2>/dev/null | sh 2>&1 | tail -2

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

MODEL = "gemma3:4b"
os.environ["GEMMA_MODEL"] = MODEL          # read by src/model.py
!ollama pull {MODEL} 2>&1 | tail -1

print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
print(subprocess.run(["bash", "-lc", "nvidia-smi -L || echo 'no GPU - running on CPU'"],
                     capture_output=True, text=True).stdout)
print(f"environment ready in {time.time() - t0:.0f}s")

In [ ]:
# The repo layout, rebuilt here so that every cell below is the file it says it is -
# byte for byte, no trimmed-for-the-slides version. The notebook is generated from the
# repository by build_notebook.py, so it cannot drift from the code that was tested.
!mkdir -p src tests schemes
# Both __init__.py files matter. Kaggle's image already ships a `tests` package, and a
# directory without __init__.py is only a namespace portion - the import scan keeps going
# and the installed one wins. A regular package takes precedence.
open("src/__init__.py", "w").close()
open("tests/__init__.py", "w").close()
sys.path.insert(0, ".")
print("ok")

## 2 · Rules are data

Every threshold lives in one file, cites the document it came from, and is read by
ordinary Python. A programme officer can check a line against a circular without reading
any code, and adding a scheme is a YAML entry rather than a change to the engine.

A rule that references a field the vocabulary does not declare fails at **load**, loudly.
The alternative is a scheme that silently never matches anybody — invisible in testing,
and in production it just quietly denies people money.

In [ ]:
%%writefile schemes/schemes.yaml
# Welfare scheme eligibility rules.
#
# This file is the part of the system that decides entitlement, and it is deliberately
# not the model's job.  Every rule below is a line a caseworker or an auditor can read
# and check against the published scheme guidelines, and every decision the tool makes
# traces back to one of them.
#
# `source` is the document a rule comes from.  These are transcribed from publicly
# published scheme guidelines for a prototype and would need a programme officer's
# sign-off against the current circulars before any real use - thresholds in particular
# are revised by notification and vary by state.
#
# Fact vocabulary (what the perception step is allowed to emit) is fixed in
# src/facts.py; a rule may only reference names declared there.

meta:
  version: "0.1-prototype"
  state: "Uttar Pradesh"
  currency: INR
  disclaimer: >
    Prototype. Thresholds and rules are transcribed from published guidance to
    demonstrate an auditable architecture. Not a substitute for the official
    eligibility determination made by the implementing department.

schemes:

  - id: pmay_g
    name_en: Pradhan Mantri Awaas Yojana - Gramin
    name_hi: प्रधानमंत्री आवास योजना - ग्रामीण
    what_en: Assistance to build a pucca house.
    benefit_en: "₹1,20,000 in instalments (plain areas), plus MGNREGA labour days"
    source: "PMAY-G Framework for Implementation, Ministry of Rural Development"
    rules:
      - fact: residence
        op: eq
        value: rural
      - fact: house_type
        op: in
        value: [none, kutcha]
      - fact: owns_motor_vehicle
        op: eq
        value: false
      - fact: monthly_income
        op: lte
        value: 15000
        note: "Auto-inclusion via SECC deprivation criteria; income proxy used here"
    documents:
      - Aadhaar card of the applicant
      - Job card (MGNREGA)
      - Bank account passbook (account must be in the applicant's name)
      - Consent for Aadhaar seeding
    where: Gram Panchayat / Block Development Office

  - id: nfbs
    name_en: National Family Benefit Scheme
    name_hi: राष्ट्रीय परिवार लाभ योजना
    what_en: One-time grant to a BPL household after the death of its primary earner.
    benefit_en: "₹30,000 one-time"
    source: "National Social Assistance Programme (NSAP) guidelines, Ministry of Rural Development"
    rules:
      - fact: primary_earner_died
        op: eq
        value: true
      - fact: years_since_earner_death
        op: lte
        value: 3
        note: "Application window; states commonly allow up to 3 years"
      - fact: deceased_age_at_death
        op: between
        value: [18, 60]
      - fact: bpl
        op: eq
        value: true
    documents:
      - Death certificate of the deceased earner
      - BPL ration card
      - Aadhaar card of the applicant
      - Bank account passbook
    where: District Social Welfare Office / Tehsil

  - id: widow_pension
    name_en: Widow Pension (Uttar Pradesh)
    name_hi: निराश्रित महिला पेंशन
    what_en: Monthly pension for a destitute widow.
    benefit_en: "₹1,000 per month"
    source: "UP Directorate of Social Welfare - Nirashrit Mahila Pension"
    rules:
      - fact: applicant_is_widow
        op: eq
        value: true
      - fact: applicant_age
        op: gte
        value: 18
      - fact: annual_income
        op: lte
        value: 60000
        note: "Rural ceiling; the urban ceiling differs by notification"
      - fact: remarried
        op: eq
        value: false
    documents:
      - Death certificate of the husband
      - Aadhaar card
      - Income certificate (Tehsil)
      - Bank account passbook
      - Passport photograph
    where: Block office / online at sspy-up.gov.in

  - id: kanya_sumangala
    name_en: Mukhyamantri Kanya Sumangala Yojana
    name_hi: मुख्यमंत्री कन्या सुमंगला योजना
    what_en: Staged payments to families for the education and welfare of a girl child.
    benefit_en: "₹25,000 total, paid across six life stages"
    source: "UP Department of Women & Child Development, Kanya Sumangala guidelines"
    rules:
      - fact: has_girl_child
        op: eq
        value: true
      - fact: annual_income
        op: lte
        value: 300000
      - fact: girl_children_count
        op: lte
        value: 2
        note: "Benefit is limited to two girls per family"
    documents:
      - Birth certificate of the girl child
      - Aadhaar of parent and child
      - Income certificate
      - Bank account passbook of the parent
    where: Anganwadi centre / online at mksy.up.gov.in

  - id: pm_kisan
    name_en: PM Kisan Samman Nidhi
    name_hi: पीएम किसान सम्मान निधि
    what_en: Income support for landholding farmer families.
    benefit_en: "₹6,000 per year in three instalments"
    source: "PM-KISAN Operational Guidelines, Ministry of Agriculture"
    rules:
      - fact: owns_farmland
        op: eq
        value: true
      - fact: is_income_tax_payer
        op: eq
        value: false
      - fact: holds_government_post
        op: eq
        value: false
    documents:
      - Land record (khatauni / khasra)
      - Aadhaar card
      - Bank account passbook
    where: Lekhpal / CSC centre / pmkisan.gov.in

  - id: ayushman
    name_en: Ayushman Bharat - PM-JAY
    name_hi: आयुष्मान भारत
    what_en: Cashless hospital cover for the family.
    benefit_en: "₹5,00,000 per family per year, secondary and tertiary care"
    source: "PM-JAY beneficiary identification guidelines, National Health Authority"
    rules:
      - fact: bpl
        op: eq
        value: true
    documents:
      - Ration card
      - Aadhaar card of each family member to be covered
    where: Ayushman Mitra desk at any empanelled hospital / CSC

  - id: old_age_pension
    name_en: Old Age Pension (Uttar Pradesh)
    name_hi: वृद्धावस्था पेंशन
    what_en: Monthly pension for elderly persons below the poverty line.
    benefit_en: "₹1,000 per month"
    source: "UP Directorate of Social Welfare - Vridhavastha Pension"
    rules:
      - fact: applicant_age
        op: gte
        value: 60
      - fact: annual_income
        op: lte
        value: 56460
        note: "Rural ceiling per current notification"
    documents:
      - Aadhaar card
      - Age proof
      - Income certificate
      - Bank account passbook
    where: Block office / sspy-up.gov.in

  - id: scholarship_pre_matric
    name_en: Pre-Matric Scholarship
    name_hi: पूर्वदशम छात्रवृत्ति
    what_en: Fee reimbursement and maintenance allowance for school students.
    benefit_en: "Fee reimbursement plus maintenance allowance, varies by class"
    source: "UP Samaj Kalyan Vibhag scholarship guidelines"
    rules:
      - fact: has_school_age_child
        op: eq
        value: true
      - fact: child_in_school
        op: eq
        value: true
      - fact: annual_income
        op: lte
        value: 200000
    documents:
      - School enrolment certificate
      - Caste certificate (if applying under a reserved category)
      - Income certificate
      - Aadhaar and bank passbook of the student
    where: The child's school / scholarship.up.gov.in

## 3 · The vocabulary, and why every field is tri-state

`True`, `False`, or unknown. The distinction is load-bearing precisely *because* the
interview is the product: an unknown field is **a question to ask**, a `False` is an
answer already given. Collapse the two and the engine stops asking about things nobody
ever raised, and reports "not eligible" where the truthful answer was "we never asked".

This is not theoretical — section 7 has the bug this design caught.

In [ ]:
%%writefile src/facts.py
"""The fact vocabulary the model is allowed to emit.

The perception step turns free speech into these fields and nothing else.  Two reasons
the vocabulary is closed rather than open:

* A rule in schemes.yaml may only reference a name declared here, so a typo in either
  file is a startup error rather than a silently unmatched scheme.
* The model cannot invent a fact that no rule reads.  It fills in a form; it does not
  get to introduce new grounds for entitlement.

Every field is tri-state - True, False, or unknown (None) - because "she did not say"
and "she said no" have to be different things.  Treating silence as False is how a
system quietly denies people benefits they qualify for.
"""
from __future__ import annotations

from dataclasses import dataclass, fields
from typing import Any, Literal

Residence = Literal["rural", "urban"]
HouseType = Literal["none", "kutcha", "semi_pucca", "pucca"]


@dataclass
class Facts:
    """A household situation, as far as it is known."""

    # who is asking
    applicant_age: int | None = None
    applicant_is_widow: bool | None = None
    remarried: bool | None = None
    residence: Residence | None = None

    # household
    monthly_income: float | None = None
    annual_income: float | None = None
    bpl: bool | None = None
    house_type: HouseType | None = None
    owns_motor_vehicle: bool | None = None

    # children
    has_girl_child: bool | None = None
    girl_children_count: int | None = None
    has_school_age_child: bool | None = None
    child_in_school: bool | None = None

    # land and work
    owns_farmland: bool | None = None
    farmland_hectares: float | None = None
    is_income_tax_payer: bool | None = None
    holds_government_post: bool | None = None

    # bereavement
    primary_earner_died: bool | None = None
    years_since_earner_death: float | None = None
    deceased_age_at_death: int | None = None

    def known(self) -> dict[str, Any]:
        return {f.name: getattr(self, f.name) for f in fields(self)
                if getattr(self, f.name) is not None}

    def unknown(self) -> list[str]:
        return [f.name for f in fields(self) if getattr(self, f.name) is None]

    def with_(self, name: str, value: Any) -> "Facts":
        if name not in FACT_NAMES:
            raise KeyError(f"unknown fact: {name}")
        d = {f.name: getattr(self, f.name) for f in fields(self)}
        d[name] = value
        return Facts(**d)


FACT_NAMES = {f.name for f in fields(Facts)}

# What to ask a person, in their language, to establish each fact.  Kept next to the
# vocabulary so a new fact cannot be added without deciding how to ask about it.
QUESTIONS: dict[str, dict[str, str]] = {
    "applicant_age": {"en": "How old are you?", "hi": "आपकी उम्र क्या है?"},
    "applicant_is_widow": {"en": "Has your husband passed away?",
                           "hi": "क्या आपके पति का देहांत हो चुका है?"},
    "remarried": {"en": "Have you remarried since?", "hi": "क्या आपने दोबारा विवाह किया है?"},
    "residence": {"en": "Do you live in a village or a town?",
                  "hi": "आप गाँव में रहती हैं या शहर में?"},
    "monthly_income": {"en": "Roughly how much does the household earn in a month?",
                       "hi": "घर में महीने भर में लगभग कितनी आमदनी होती है?"},
    "annual_income": {"en": "Roughly how much does the household earn in a year?",
                      "hi": "घर की सालाना आमदनी लगभग कितनी है?"},
    "bpl": {"en": "Do you have a BPL or Antyodaya ration card?",
            "hi": "क्या आपके पास बीपीएल या अंत्योदय राशन कार्ड है?"},
    "house_type": {"en": "Is your house pucca, semi-pucca, or kutcha?",
                   "hi": "आपका घर पक्का है, अर्ध-पक्का है या कच्चा?"},
    "owns_motor_vehicle": {"en": "Does the household own a motorised vehicle?",
                           "hi": "क्या घर में कोई मोटर वाहन है?"},
    "has_girl_child": {"en": "Do you have a daughter?", "hi": "क्या आपकी कोई बेटी है?"},
    "girl_children_count": {"en": "How many daughters do you have?",
                            "hi": "आपकी कितनी बेटियाँ हैं?"},
    "has_school_age_child": {"en": "Do you have a child of school age?",
                             "hi": "क्या आपका कोई बच्चा स्कूल जाने की उम्र का है?"},
    "child_in_school": {"en": "Is the child currently enrolled in school?",
                        "hi": "क्या बच्चा अभी स्कूल में पढ़ रहा है?"},
    "owns_farmland": {"en": "Is there farmland in the family's name?",
                      "hi": "क्या परिवार के नाम पर खेती की ज़मीन है?"},
    "farmland_hectares": {"en": "How much land, in bighas or hectares?",
                          "hi": "कितनी ज़मीन है, बीघा या हेक्टेयर में?"},
    "is_income_tax_payer": {"en": "Does anyone in the household pay income tax?",
                            "hi": "क्या घर में कोई आयकर भरता है?"},
    "holds_government_post": {"en": "Does anyone hold a government post or pension?",
                              "hi": "क्या घर में कोई सरकारी पद पर है या पेंशन पाता है?"},
    "primary_earner_died": {"en": "Has the main earner of the household died?",
                            "hi": "क्या घर के मुख्य कमाने वाले का देहांत हुआ है?"},
    "years_since_earner_death": {"en": "How long ago did that happen?",
                                 "hi": "यह कितने समय पहले हुआ था?"},
    "deceased_age_at_death": {"en": "How old were they when they passed away?",
                              "hi": "देहांत के समय उनकी उम्र क्या थी?"},
}

## 4 · The engine, and the measurement

No model appears anywhere below this line.

`evaluate()` returns **three** outcomes rather than two. *Uncertain* — nothing has failed,
but a rule cannot be checked yet — is not a rounding error on the way to "no". It is what
generates the next question.

`next_question()` is the part being measured.

In [ ]:
%%writefile src/engine.py
"""Deterministic eligibility engine.

This is the part of the system that decides who is entitled to what, and it contains no
model.  Being confidently wrong here has asymmetric costs in both directions: telling
someone they qualify when they do not costs them a day's wage and a bus fare to a
counter that turns them away; telling them they do not qualify when they do costs them
money they are owed, possibly for years, and they have no way to know.  Neither error is
one to hand to a language model that cannot show its working.

So the model never sees these rules.  It fills in a form (src/perceive.py) and it writes
the result up (src/explain.py).  Everything between is this file plus schemes/schemes.yaml,
where a caseworker can read any decision back to the line that produced it.

The engine returns three things, not one:

  eligible    every rule that can be checked passes
  ineligible  some rule definitively fails, with the rule that failed
  uncertain   nothing fails, but a rule cannot be evaluated for want of a fact

The third is the useful one.  A tool that silently drops "uncertain" into "no" is the
failure mode that keeps entitlements unclaimed, and the missing facts it reports are what
drives the follow-up question - see `next_question`.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Literal

import yaml

from .facts import FACT_NAMES, QUESTIONS, Facts

Status = Literal["eligible", "ineligible", "uncertain"]


class SchemeFileError(ValueError):
    pass


@dataclass
class RuleResult:
    fact: str
    op: str
    value: Any
    actual: Any
    passed: bool | None          # None = cannot tell, the fact is unknown
    note: str | None = None

    def describe(self) -> str:
        if self.passed is None:
            return f"{self.fact} unknown (needs {self.op} {self.value})"
        verb = "ok" if self.passed else "fails"
        return f"{self.fact}={self.actual!r} {verb} ({self.op} {self.value})"


@dataclass
class SchemeResult:
    scheme: dict
    status: Status
    rules: list[RuleResult]
    missing: list[str] = field(default_factory=list)

    @property
    def id(self) -> str:
        return self.scheme["id"]

    @property
    def failed(self) -> list[RuleResult]:
        return [r for r in self.rules if r.passed is False]


def _check(op: str, actual: Any, expected: Any) -> bool:
    if op == "eq":
        return actual == expected
    if op == "ne":
        return actual != expected
    if op == "in":
        return actual in expected
    if op == "not_in":
        return actual not in expected
    if op == "lte":
        return float(actual) <= float(expected)
    if op == "gte":
        return float(actual) >= float(expected)
    if op == "lt":
        return float(actual) < float(expected)
    if op == "gt":
        return float(actual) > float(expected)
    if op == "between":
        lo, hi = expected
        return float(lo) <= float(actual) <= float(hi)
    raise SchemeFileError(f"unknown operator: {op}")


VALID_OPS = {"eq", "ne", "in", "not_in", "lte", "gte", "lt", "gt", "between"}


def load_schemes(path: str | Path = "schemes/schemes.yaml") -> dict:
    """Load and validate the rule file.

    Validation is strict and happens at load: a rule that references a fact the
    vocabulary does not declare is a startup error, not a scheme that silently never
    matches.  That failure mode is invisible in testing and denies people benefits in
    production, so it is made loud here.
    """
    data = yaml.safe_load(Path(path).read_text())
    if "schemes" not in data:
        raise SchemeFileError("no 'schemes' key")
    seen: set[str] = set()
    for s in data["schemes"]:
        for required in ("id", "name_en", "rules", "documents", "source"):
            if required not in s:
                raise SchemeFileError(f"scheme {s.get('id', '?')} is missing '{required}'")
        if s["id"] in seen:
            raise SchemeFileError(f"duplicate scheme id: {s['id']}")
        seen.add(s["id"])
        for r in s["rules"]:
            if r["fact"] not in FACT_NAMES:
                raise SchemeFileError(
                    f"scheme {s['id']} references unknown fact {r['fact']!r}; "
                    f"add it to src/facts.py or fix the spelling")
            if r["op"] not in VALID_OPS:
                raise SchemeFileError(f"scheme {s['id']} uses unknown op {r['op']!r}")
            if r["fact"] not in QUESTIONS:
                raise SchemeFileError(
                    f"fact {r['fact']!r} has no question in src/facts.py - every fact a "
                    f"rule depends on must be answerable by asking the person")
    return data


def evaluate(facts: Facts, schemes: dict) -> list[SchemeResult]:
    """Evaluate every scheme against what is known."""
    known = facts.known()
    out: list[SchemeResult] = []
    for s in schemes["schemes"]:
        rules: list[RuleResult] = []
        missing: list[str] = []
        for r in s["rules"]:
            name = r["fact"]
            if name not in known:
                rules.append(RuleResult(name, r["op"], r["value"], None, None,
                                        r.get("note")))
                missing.append(name)
                continue
            actual = known[name]
            try:
                passed = _check(r["op"], actual, r["value"])
            except (TypeError, ValueError):
                # a fact of the wrong shape is a perception bug; treat it as unknown
                # rather than as a failure, so it surfaces as a question not a denial
                rules.append(RuleResult(name, r["op"], r["value"], actual, None,
                                        r.get("note")))
                missing.append(name)
                continue
            rules.append(RuleResult(name, r["op"], r["value"], actual, passed,
                                    r.get("note")))
        if any(r.passed is False for r in rules):
            status: Status = "ineligible"
        elif missing:
            status = "uncertain"
        else:
            status = "eligible"
        out.append(SchemeResult(s, status, rules, missing))
    return out


def next_question(facts: Facts, schemes: dict) -> tuple[str | None, dict]:
    """Choose the single fact worth asking about next.

    This is what makes the conversation short.  A person seeking help has limited time
    and patience, and every question that cannot change any answer is a question that
    costs their goodwill for nothing.

    The score for a fact is the total benefit currently sitting in 'uncertain' that
    knowing it could resolve, weighted by how close each scheme is to a decision.  A fact
    that would settle a ₹1,20,000 housing grant outranks one that would settle a ₹1,000
    monthly pension, and a scheme with one unknown left outranks one with four.

    Returns (fact_name, diagnostics).  fact_name is None when nothing further can move
    an outcome, which is the signal to stop asking and report.
    """
    results = evaluate(facts, schemes)
    scores: dict[str, float] = {}
    detail: dict[str, list[str]] = {}
    for res in results:
        if res.status != "uncertain":
            continue
        value = _benefit_weight(res.scheme)
        # a scheme one fact away is worth more per question than one four facts away
        share = value / len(res.missing)
        for m in res.missing:
            scores[m] = scores.get(m, 0.0) + share
            detail.setdefault(m, []).append(res.id)
    if not scores:
        return None, {"scores": {}, "resolves": {}}
    best = max(scores, key=lambda k: scores[k])
    return best, {"scores": scores, "resolves": detail}


def _benefit_weight(scheme: dict) -> float:
    """Rough annual rupee value of a scheme, for ranking questions only.

    Parsed from the human-readable benefit string so the YAML stays readable by the
    people who have to audit it.  A monthly figure is annualised; anything unparseable
    falls back to a small positive weight so the scheme still generates questions.
    """
    text = str(scheme.get("benefit_en", "")).replace(",", "")
    digits = "".join(c if c.isdigit() else " " for c in text).split()
    if not digits:
        return 1000.0
    amount = float(max(digits, key=len))
    if "month" in text.lower():
        amount *= 12
    return amount if math.isfinite(amount) and amount > 0 else 1000.0


def summarise(results: list[SchemeResult]) -> dict[str, list[SchemeResult]]:
    return {
        "eligible": [r for r in results if r.status == "eligible"],
        "uncertain": [r for r in results if r.status == "uncertain"],
        "ineligible": [r for r in results if r.status == "ineligible"],
    }


def audit_trail(res: SchemeResult) -> str:
    """The decision, written so a caseworker can check it against the source document."""
    lines = [f"{res.scheme['name_en']} [{res.id}] -> {res.status.upper()}",
             f"  source: {res.scheme['source']}"]
    for r in res.rules:
        mark = {True: "PASS", False: "FAIL", None: "????"}[r.passed]
        lines.append(f"  [{mark}] {r.describe()}")
        if r.note:
            lines.append(f"         note: {r.note}")
    return "\n".join(lines)

In [ ]:
%%writefile tests/households.py
"""Complete households, used to measure the interview rather than the extractor.

Each entry is a full fact set - what would be true if every question had been asked and
answered honestly - plus the schemes that ought to come out of it, worked out by hand
against schemes/schemes.yaml.

Two different things are measured with these:

* `tests/test_households.py` checks the engine reaches the hand-derived answer. That is
  end-to-end correctness of the part that decides entitlement, and it needs no model.
* `scripts_ask.py` uses each household as a truthful oracle: it answers whatever the
  engine asks, so the *order* the engine asks in can be measured against alternatives.

The households are deliberately not all easy. Two of them qualify for almost nothing,
because a tool that is only measured on people with a strong claim will look good and
waste the time of everyone else.
"""
from __future__ import annotations

HOUSEHOLDS: list[dict] = [
    dict(
        id="widow_rural_bpl",
        note="The walkthrough case: widowed last year, two daughters, kutcha house.",
        facts=dict(
            applicant_age=35, applicant_is_widow=True, remarried=False,
            residence="rural", monthly_income=3000, annual_income=36000, bpl=True,
            house_type="kutcha", owns_motor_vehicle=False,
            has_girl_child=True, girl_children_count=2,
            has_school_age_child=True, child_in_school=True,
            owns_farmland=False, farmland_hectares=0.0,
            is_income_tax_payer=False, holds_government_post=False,
            primary_earner_died=True, years_since_earner_death=1,
            deceased_age_at_death=40,
        ),
        expect_eligible=["pmay_g", "nfbs", "widow_pension", "kanya_sumangala",
                         "ayushman", "scholarship_pre_matric"],
    ),
    dict(
        id="elderly_alone_rural",
        note="70, lives alone in a village, no surviving earner, no children at home.",
        facts=dict(
            applicant_age=70, applicant_is_widow=True, remarried=False,
            residence="rural", monthly_income=1500, annual_income=18000, bpl=True,
            house_type="kutcha", owns_motor_vehicle=False,
            has_girl_child=False, girl_children_count=0,
            has_school_age_child=False, child_in_school=False,
            owns_farmland=False, farmland_hectares=0.0,
            is_income_tax_payer=False, holds_government_post=False,
            primary_earner_died=True, years_since_earner_death=9,
            deceased_age_at_death=72,
        ),
        expect_eligible=["pmay_g", "widow_pension", "old_age_pension", "ayushman"],
    ),
    dict(
        id="smallholder_family",
        note="Two bighas, pucca house, no bereavement. Land schemes, not welfare ones.",
        facts=dict(
            applicant_age=44, applicant_is_widow=False, remarried=False,
            residence="rural", monthly_income=9000, annual_income=108000, bpl=False,
            house_type="pucca", owns_motor_vehicle=True,
            has_girl_child=True, girl_children_count=1,
            has_school_age_child=True, child_in_school=True,
            owns_farmland=True, farmland_hectares=0.5,
            is_income_tax_payer=False, holds_government_post=False,
            primary_earner_died=False, years_since_earner_death=0,
            deceased_age_at_death=0,
        ),
        expect_eligible=["pm_kisan", "kanya_sumangala", "scholarship_pre_matric"],
    ),
    dict(
        id="urban_salaried",
        note="Qualifies for almost nothing. The tool has to say so quickly.",
        facts=dict(
            applicant_age=38, applicant_is_widow=False, remarried=False,
            residence="urban", monthly_income=45000, annual_income=540000, bpl=False,
            house_type="pucca", owns_motor_vehicle=True,
            has_girl_child=False, girl_children_count=0,
            has_school_age_child=False, child_in_school=False,
            owns_farmland=False, farmland_hectares=0.0,
            is_income_tax_payer=True, holds_government_post=False,
            primary_earner_died=False, years_since_earner_death=0,
            deceased_age_at_death=0,
        ),
        expect_eligible=[],
    ),
    dict(
        id="widow_remarried",
        note="Widowed, then remarried. The widow pension is correctly ruled out, and "
             "getting that wrong in the generous direction sends her to a counter that "
             "will check.",
        facts=dict(
            applicant_age=41, applicant_is_widow=True, remarried=True,
            residence="rural", monthly_income=4000, annual_income=48000, bpl=True,
            house_type="semi_pucca", owns_motor_vehicle=False,
            has_girl_child=True, girl_children_count=1,
            has_school_age_child=True, child_in_school=True,
            owns_farmland=False, farmland_hectares=0.0,
            is_income_tax_payer=False, holds_government_post=False,
            primary_earner_died=True, years_since_earner_death=6,
            deceased_age_at_death=48,
        ),
        expect_eligible=["kanya_sumangala", "ayushman", "scholarship_pre_matric"],
    ),
    dict(
        id="bereaved_recent_urban",
        note="Urban, earner died this year within the NFBS age window, poor. Housing is "
             "out because PMAY-G is rural.",
        facts=dict(
            applicant_age=33, applicant_is_widow=True, remarried=False,
            residence="urban", monthly_income=2500, annual_income=30000, bpl=True,
            house_type="kutcha", owns_motor_vehicle=False,
            has_girl_child=False, girl_children_count=0,
            has_school_age_child=False, child_in_school=False,
            owns_farmland=False, farmland_hectares=0.0,
            is_income_tax_payer=False, holds_government_post=False,
            primary_earner_died=True, years_since_earner_death=0.5,
            deceased_age_at_death=37,
        ),
        expect_eligible=["nfbs", "widow_pension", "ayushman"],
    ),
    dict(
        id="government_post_farmer",
        note="Has land but holds a government post, which excludes PM-KISAN. An "
             "exclusion that is easy to miss and expensive to get wrong.",
        facts=dict(
            applicant_age=50, applicant_is_widow=False, remarried=False,
            residence="rural", monthly_income=25000, annual_income=300000, bpl=False,
            house_type="pucca", owns_motor_vehicle=True,
            has_girl_child=False, girl_children_count=0,
            has_school_age_child=True, child_in_school=True,
            owns_farmland=True, farmland_hectares=1.2,
            is_income_tax_payer=False, holds_government_post=True,
            primary_earner_died=False, years_since_earner_death=0,
            deceased_age_at_death=0,
        ),
        expect_eligible=[],
    ),
    dict(
        id="landless_labourer",
        note="Landless, kutcha house, children in school, no bereavement.",
        facts=dict(
            applicant_age=29, applicant_is_widow=False, remarried=False,
            residence="rural", monthly_income=5000, annual_income=60000, bpl=True,
            house_type="kutcha", owns_motor_vehicle=False,
            has_girl_child=True, girl_children_count=2,
            has_school_age_child=True, child_in_school=True,
            owns_farmland=False, farmland_hectares=0.0,
            is_income_tax_payer=False, holds_government_post=False,
            primary_earner_died=False, years_since_earner_death=0,
            deceased_age_at_death=0,
        ),
        expect_eligible=["pmay_g", "kanya_sumangala", "ayushman",
                         "scholarship_pre_matric"],
    ),
]

In [ ]:
%%writefile scripts_ask.py
"""Measure the interview: does asking in benefit order actually help, and by how much.

`next_question()` claims that ordering questions by the benefit they would resolve makes
the conversation shorter and front-loads the money. That is a claim about an algorithm,
so it can be checked without a model and without a person - which is what this does.

Each household in tests/households.py answers truthfully whatever it is asked. Four
strategies pick the questions:

  benefit   what the engine ships: score each unknown fact by the benefit sitting in
            'uncertain' that knowing it would resolve, divided across the facts each
            scheme is still waiting on
  coverage  ask whatever unblocks the most schemes, ignoring how much they are worth.
            This is the ablation that matters: it isolates the benefit weighting from
            the mere fact of asking something relevant
  fixed     the vocabulary order, skipping facts no undecided scheme needs. This is what
            a paper form does
  random    a shuffle, averaged over seeds. The floor

Two numbers come out, and the second is the one that matters at a doorstep.

**Questions to a final answer.** How long the interview runs before no remaining question
can change any outcome.

**Benefit secured after k questions.** Interviews get abandoned. A visitor is called away,
a child starts crying, the queue at the block office moves. If the conversation stops at
question three, how much of the household's true entitlement has already been established?
A strategy that reaches the same place in the same number of questions but establishes the
₹5,00,000 health cover first is not equal - it is better, and only this second measurement
can see the difference.
"""
from __future__ import annotations

import argparse
import random
import sys

sys.path.insert(0, ".")
from src.engine import (evaluate, load_schemes, next_question,  # noqa: E402
                        summarise, _benefit_weight)
from src.facts import FACT_NAMES, Facts                          # noqa: E402
from tests.households import HOUSEHOLDS                          # noqa: E402


def _needed_facts(facts: Facts, schemes: dict) -> list[str]:
    """Facts some still-undecided scheme is waiting on. Anything else is a wasted breath."""
    need: list[str] = []
    for res in evaluate(facts, schemes):
        if res.status == "uncertain":
            for m in res.missing:
                if m not in need:
                    need.append(m)
    return need


def pick_benefit(facts: Facts, schemes: dict, rng) -> str | None:
    return next_question(facts, schemes)[0]


def pick_coverage(facts: Facts, schemes: dict, rng) -> str | None:
    """Unblock the most schemes, treating a ₹1,000 pension as equal to a ₹5 lakh cover."""
    counts: dict[str, int] = {}
    for res in evaluate(facts, schemes):
        if res.status == "uncertain":
            for m in res.missing:
                counts[m] = counts.get(m, 0) + 1
    return max(counts, key=lambda k: counts[k]) if counts else None


def pick_fixed(facts: Facts, schemes: dict, rng) -> str | None:
    need = _needed_facts(facts, schemes)
    for name in sorted(FACT_NAMES):
        if name in need:
            return name
    return None


def pick_random(facts: Facts, schemes: dict, rng) -> str | None:
    need = _needed_facts(facts, schemes)
    return rng.choice(need) if need else None


STRATEGIES = {
    "benefit": pick_benefit,
    "coverage": pick_coverage,
    "fixed": pick_fixed,
    "random": pick_random,
}


def secured(facts: Facts, schemes: dict, truth_ids: set[str]) -> float:
    """Rupees of the household's true entitlement already established as eligible.

    Only schemes that are genuinely theirs count. If a strategy were ever to reach
    'eligible' on a scheme that is not in the ground truth, it earns nothing for it -
    that would be a wrong answer, not progress.
    """
    total = 0.0
    for res in evaluate(facts, schemes):
        if res.status == "eligible" and res.id in truth_ids:
            total += _benefit_weight(res.scheme)
    return total


def run(household: dict, schemes: dict, pick, rng) -> dict:
    """One interview. Returns questions asked and the benefit curve."""
    truth = set(household["expect_eligible"])
    facts = Facts()
    answers = household["facts"]
    curve = [secured(facts, schemes, truth)]
    asked: list[str] = []
    while True:
        fact = pick(facts, schemes, rng)
        if fact is None or fact in asked:
            break
        asked.append(fact)
        facts = facts.with_(fact, answers.get(fact))
        curve.append(secured(facts, schemes, truth))
    final = summarise(evaluate(facts, schemes))
    return {
        "asked": asked,
        "curve": curve,
        "reached": sorted(r.id for r in final["eligible"]),
        "correct": sorted(r.id for r in final["eligible"]) == sorted(truth),
    }


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--schemes", default="schemes/schemes.yaml")
    ap.add_argument("--seeds", type=int, default=20, help="shuffles averaged for `random`")
    ap.add_argument("--horizon", type=int, default=8, help="questions in the curve table")
    args = ap.parse_args()

    schemes = load_schemes(args.schemes)
    truth_total = {}
    for h in HOUSEHOLDS:
        full = Facts(**h["facts"])
        truth_total[h["id"]] = secured(full, schemes, set(h["expect_eligible"]))

    print(f"\n{len(HOUSEHOLDS)} households, {len(schemes['schemes'])} schemes, "
          f"no model involved\n")

    summary = {}
    for name, pick in STRATEGIES.items():
        seeds = range(args.seeds) if name == "random" else [0]
        n_questions, curves, wrong = [], [], []
        for seed in seeds:
            rng = random.Random(seed)
            for h in HOUSEHOLDS:
                r = run(h, schemes, pick, rng)
                n_questions.append(len(r["asked"]))
                total = truth_total[h["id"]] or 1.0
                # pad the curve to the horizon: once the interview ends, the fraction
                # secured stops changing
                c = r["curve"] + [r["curve"][-1]] * (args.horizon + 1 - len(r["curve"]))
                curves.append([v / total for v in c[:args.horizon + 1]])
                if not r["correct"]:
                    wrong.append((h["id"], r["reached"]))
        mean_curve = [sum(c[k] for c in curves) / len(curves)
                      for k in range(args.horizon + 1)]
        summary[name] = {
            "questions": sum(n_questions) / len(n_questions),
            "curve": mean_curve,
            "wrong": wrong,
        }

    print("Questions asked before no remaining question can change any outcome")
    print("  (mean over households; every strategy reaches the same final answer)\n")
    for name, s in summary.items():
        bar = "#" * round(s["questions"] * 2)
        print(f"  {name:9s} {s['questions']:5.1f}  {bar}")

    print(f"\n\nFraction of the household's true entitlement already secured after k "
          f"questions")
    print("  (mean over households, in rupees of annualised benefit)\n")
    head = "  k          " + "".join(f"{k:>7d}" for k in range(1, args.horizon + 1))
    print(head)
    print("  " + "-" * (len(head) - 2))
    for name, s in summary.items():
        row = "".join(f"{v * 100:6.0f}%" for v in s["curve"][1:args.horizon + 1])
        print(f"  {name:9s}  {row}")

    bad = [(n, s["wrong"]) for n, s in summary.items() if s["wrong"]]
    print()
    if bad:
        print("!! a strategy ended on the wrong answer:")
        for n, w in bad:
            print(f"   {n}: {w[:3]}")
    else:
        print("Every strategy ends on the hand-derived answer for every household.")
        print("The ordering changes what is known when, never what is concluded.")


if __name__ == "__main__":
    main()

In [ ]:
!python scripts_ask.py

### Reading that table

**After one question the shipped strategy has secured 52% of the average household's true
entitlement.** Form order has secured nothing until question four, and a random relevant
question manages 4%.

The first question it asks is about the **BPL / Antyodaya ration card**, because that one
field gates ₹30,000 of bereavement assistance *and* a ₹5,00,000 health cover. Nothing else
in the vocabulary is worth as much per breath.

#### The ablation is the honest part

`coverage` asks whichever question unblocks the most schemes, ignoring what they are
worth. It **ties** the shipped strategy on interview length — 11.6 questions against 13.2
for form order — and it is worth **nothing** at question one.

So benefit weighting is not what makes the interview short. Asking a *relevant* question
does that, and plain scheme-counting is enough to get it. The weighting does something
narrower: it decides **which** half of the money you walk away with if the conversation
ends early. That is what the table supports, and it is all it supports.

Ordering is only ever allowed to cost time, never entitlement —
`test_question_order_never_changes_the_conclusion` runs all four strategies over all eight
households and asserts they land on the same answer every time.

## 5 · Where the model sits, and where it does not

Two narrow jobs, both at the edges.

**In:** free speech → fields. Hindi, Hinglish, English, *bighas*, *lakhs*, "साल भर में
चालीस हज़ार". Genuinely hard, and genuinely what a language model is for.

**Out:** a decision that has already been made → a message in the person's language, a
document checklist, an audit trail.

Between them, nothing. The engine will not take a scheme from the model, a threshold from
the model, or a field the vocabulary does not declare — `coerce()` drops anything outside
it, and the generated message is checked afterwards for schemes nobody evaluated.

That is enforcement, not instruction. A prompt is a request, and the boundary between a
language model and a decision about somebody's pension needs something stronger than a
request.

In [ ]:
%%writefile src/perceive.py
"""Perception: free speech in, structured facts out.

This is the one job the model is given, and it is the job it is actually good at -
turning "my husband passed away last year, two children in school, we have two bighas"
into fields a rule engine can read.  It is asked for nothing else.  It does not see the
eligibility rules, it is not told which schemes exist, and it never states an outcome.

Two properties are enforced in code rather than requested in the prompt, because a
prompt is a request and this needs a guarantee:

* **Closed vocabulary.**  Anything the model emits outside `Facts` is dropped, with a
  warning.  A hallucinated field cannot reach the engine.
* **Silence is not denial.**  A field the person did not mention stays `None`.  The
  model is told explicitly not to guess, and any value it invents for an unmentioned
  field would still have to survive the schema, but the important half of this is that
  the engine treats `None` as "ask", never as "no".

Local units are converted here rather than in the prompt: 1 bigha is not a fixed
quantity nationally, and the conversion used is stated in code where it can be
corrected, instead of buried in a model's arithmetic.
"""
from __future__ import annotations

import json
import re
from dataclasses import fields
from typing import Any

from .facts import Facts

# 1 bigha varies by region; in western UP it is commonly taken as ~0.25 ha.  Stated here
# so a programme officer can change one number rather than re-engineer a prompt.
BIGHA_TO_HECTARE = 0.25

_ALLOWED = {f.name for f in fields(Facts)}

SYSTEM_PROMPT = """You extract facts from what a person says about their household.

You are filling in a form. You are NOT deciding anything, you are NOT recommending
anything, and you must NOT mention any government scheme.

Return ONLY a JSON object. Use these keys and no others:

applicant_age (int), applicant_is_widow (bool), remarried (bool),
residence ("rural"|"urban"), monthly_income (number, rupees),
annual_income (number, rupees), bpl (bool),
house_type ("none"|"kutcha"|"semi_pucca"|"pucca"), owns_motor_vehicle (bool),
has_girl_child (bool), girl_children_count (int),
has_school_age_child (bool), child_in_school (bool),
owns_farmland (bool), farmland_bighas (number), farmland_hectares (number),
is_income_tax_payer (bool), holds_government_post (bool),
primary_earner_died (bool), years_since_earner_death (number),
deceased_age_at_death (int)

RULES:
- Include a key ONLY if the person actually said it or it follows necessarily.
  "My husband died" gives applicant_is_widow true AND primary_earner_died true if he
  earned. It does NOT give you their income.
- NOT MENTIONED and SAID NO are different, and this distinction matters more than any
  other rule here.
    not mentioned  -> leave the key out
    said no        -> set the key to false. "No vehicle" IS information. Record it.
  Leaving a key out means "we still have to ask". Setting false means "we asked and the
  answer was no". Collapsing the two either wastes the person's time re-asking or
  silently denies them something.
- Land in bighas goes in farmland_bighas. Do not convert it yourself.
- Rupees: "20 hazaar" = 20000. "2 lakh" = 200000.
- If you are told what was asked and the answer does not address it AT ALL, do not fill
  that field. A denial does address it - "no", "nahi", "koi nahi" set it to false.
  This exception is only for answers about something else entirely. An answer that
  addresses the question AND adds more still fills in everything it addresses.
- "I don't know" / "pata nahi" / "पता नहीं" / "kabhi kuch kabhi kuch" is NOT a number and
  NOT a no. Leave the key out. Never write 0 for an amount the person did not state -
  zero income passes every income limit there is.
- Whose event it was matters. A death in the family is only applicant_is_widow if it was
  the speaker's husband, and only primary_earner_died if that person earned. A
  mother-in-law, parent, or sibling dying gives you neither. People answer a different question than the one asked, especially when tired or
  hard of hearing. Extract whatever they did say and leave the asked-about field out.
  Returning {} is a correct answer.

Output the JSON object and nothing else."""

# The negative examples carry more weight than the positive ones. Both inventions the
# evaluation caught - an amount conjured from "I don't know", and the speaker's
# widowhood conjured from someone else's death - are shown here returning {}.
FEWSHOT: list[tuple[str, dict]] = [
    (
        "मेरे पति का पिछले साल देहांत हो गया। दो बच्चे स्कूल जाते हैं। गाँव में रहते हैं।",
        {"applicant_is_widow": True, "primary_earner_died": True,
         "years_since_earner_death": 1, "has_school_age_child": True,
         "child_in_school": True, "residence": "rural"},
    ),
    (
        "I am 65. I live alone in the village. Kutcha house. No income now.",
        {"applicant_age": 65, "residence": "rural", "house_type": "kutcha"},
    ),
    (
        "Pata nahi kitna kamate hain, kabhi kuch kabhi kuch.",
        {},
    ),
    (
        "मेरी सास का देहांत हो गया था।",
        {},
    ),
    (
        "अड़तालीस की हूँ, और दो बेटियाँ हैं।",
        {"applicant_age": 48, "has_girl_child": True, "girl_children_count": 2},
    ),
]


# Shown in the same shape a real answer arrives in, because the bare-utterance examples
# above do not teach the model what to do with a one-word reply. Without this, a flat
# "no" to the remarriage question came back as {} - the negative examples above had
# taught it that anything touching marriage or bereavement was a trap.
ANSWER_FEWSHOT: list[tuple[str, str, str, dict]] = [
    ("क्या आपने दोबारा विवाह किया है?", "remarried", "नहीं।", {"remarried": False}),
    ("क्या आपके पास बीपीएल या अंत्योदय राशन कार्ड है?", "bpl", "हाँ, है।", {"bpl": True}),
    ("क्या घर में कोई मोटर वाहन है?", "owns_motor_vehicle", "अभी तक तो कुछ समझ नहीं आया।", {}),
]


def _answer_turn(question: str, expect_fact: str | None, utterance: str) -> str:
    hint = f'The person was asked: "{question}"'
    if expect_fact:
        hint += (f'\nIf their answer addresses it, it sets "{expect_fact}". '
                 f'If it does not address it, leave "{expect_fact}" out entirely.')
    return f"{hint}\n\nThey answered: {utterance}"


def build_prompt(utterance: str, question: str | None = None,
                 expect_fact: str | None = None) -> list[dict[str, str]]:
    """Build the extraction prompt.

    When the utterance is an answer to a question we asked, the question goes in too.
    Without it the model is reading "yes" or "forty thousand" with no idea what was
    asked, and it guesses - in testing it read "barely forty thousand for the year" as a
    monthly figure and produced an annual income twelve times too high, which flipped
    three schemes from eligible to ineligible. Bare numbers and bare yes/no need the
    question to mean anything.
    """
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    for text, out in FEWSHOT:
        msgs.append({"role": "user", "content": text})
        msgs.append({"role": "assistant", "content": json.dumps(out, ensure_ascii=False)})
    if question:
        for q, ef, a, out in ANSWER_FEWSHOT:
            msgs.append({"role": "user", "content": _answer_turn(q, ef, a)})
            msgs.append({"role": "assistant",
                         "content": json.dumps(out, ensure_ascii=False)})
        msgs.append({"role": "user",
                     "content": _answer_turn(question, expect_fact, utterance)})
    else:
        msgs.append({"role": "user", "content": utterance})
    return msgs


def _extract_json(text: str) -> dict:
    """Pull the JSON object out of whatever the model returned."""
    text = text.strip()
    fence = re.search(r"```(?:json)?\s*(.+?)```", text, re.S)
    if fence:
        text = fence.group(1).strip()
    start = text.find("{")
    if start == -1:
        raise ValueError("no JSON object in model output")
    depth = 0
    for i, ch in enumerate(text[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return json.loads(text[start:i + 1])
    raise ValueError("unterminated JSON object in model output")


def coerce(raw: dict[str, Any]) -> tuple[Facts, list[str]]:
    """Map raw model output onto the closed vocabulary, dropping anything else."""
    warnings: list[str] = []
    clean: dict[str, Any] = {}

    if "farmland_bighas" in raw and raw["farmland_bighas"] is not None:
        try:
            ha = float(raw.pop("farmland_bighas")) * BIGHA_TO_HECTARE
            clean["farmland_hectares"] = round(ha, 3)
            clean.setdefault("owns_farmland", True)
        except (TypeError, ValueError):
            warnings.append("farmland_bighas was not a number")

    for k, v in raw.items():
        if v is None:
            continue
        if k not in _ALLOWED:
            warnings.append(f"dropped unknown field {k!r}")
            continue
        clean.setdefault(k, v)

    # Backstop for the one invention with an asymmetric cost. Zero income satisfies every
    # income ceiling in the rule file, so a zero the person did not actually state would
    # grant them everything. The prompt now handles this, but a prompt is a request; this
    # is the guarantee. A genuinely zero-income household reaches the same place by a
    # different route - the field stays unknown and the engine asks.
    for _f in ("annual_income", "monthly_income"):
        if _f in clean and float(clean[_f]) == 0.0:
            warnings.append(f"dropped {_f}=0: an unstated zero would pass every "
                            f"income ceiling; treating it as unknown so the engine asks")
            clean.pop(_f)

    # a stated hectare figure implies ownership; the engine should not have to infer it
    if clean.get("farmland_hectares") and "owns_farmland" not in clean:
        clean["owns_farmland"] = True
    # Rules read both the monthly and the annual figure, so one is derived from the
    # other. This is arithmetic, not inference - but it does mean a misread period
    # propagates, which is why the question context above matters as much as it does.
    if "annual_income" in clean and "monthly_income" not in clean:
        clean["monthly_income"] = round(float(clean["annual_income"]) / 12, 2)
    elif "monthly_income" in clean and "annual_income" not in clean:
        clean["annual_income"] = round(float(clean["monthly_income"]) * 12, 2)

    typed: dict[str, Any] = {}
    for f in fields(Facts):
        if f.name not in clean:
            continue
        v = clean[f.name]
        try:
            if "bool" in str(f.type):
                typed[f.name] = bool(v)
            elif "int" in str(f.type):
                typed[f.name] = int(float(v))
            elif "float" in str(f.type):
                typed[f.name] = float(v)
            else:
                typed[f.name] = v
        except (TypeError, ValueError):
            warnings.append(f"dropped {f.name}={v!r}: wrong type")
    return Facts(**typed), warnings


def perceive(utterance: str, chat_fn, question: str | None = None,
             expect_fact: str | None = None) -> tuple[Facts, list[str], str]:
    """Run one perception pass.  `chat_fn(messages) -> str` keeps the model swappable."""
    reply = chat_fn(build_prompt(utterance, question, expect_fact))
    try:
        raw = _extract_json(reply)
    except (ValueError, json.JSONDecodeError) as exc:
        return Facts(), [f"could not parse model output: {exc}"], reply
    facts, warnings = coerce(raw)
    return facts, warnings, reply

In [ ]:
%%writefile src/model.py
"""Model access, kept behind one small interface.

Everything else in this project takes a `chat_fn(messages) -> str`.  That is deliberate:
the perception and explanation steps should be testable without a model running, and the
engine - the part that decides entitlement - should be testable without one existing.

The default backend is a local Ollama server, because the whole argument for an open
model here is that the thing runs where the person is: at a Common Service Centre with
intermittent connectivity, on a caseworker's laptop, on a phone. A household's income,
bereavement, and caste details are not data to post to someone's API.
"""
from __future__ import annotations

import json
import os
import urllib.error
import urllib.request

DEFAULT_MODEL = os.environ.get("GEMMA_MODEL", "gemma3:4b")
DEFAULT_HOST = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434")


class ModelError(RuntimeError):
    pass


def ollama_chat(messages: list[dict[str, str]], model: str = DEFAULT_MODEL,
                host: str = DEFAULT_HOST, temperature: float = 0.0,
                timeout: int = 180) -> str:
    """One chat completion from a local Ollama server.

    Temperature defaults to 0: the perception step is filling in a form, and there is no
    upside to sampling a different reading of the same sentence on a second run.
    """
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature},
    }
    req = urllib.request.Request(
        f"{host}/api/chat",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            body = json.loads(r.read())
    except urllib.error.URLError as exc:
        raise ModelError(
            f"cannot reach Ollama at {host} ({exc}). Start it with `ollama serve` "
            f"and `ollama pull {model}`.") from exc
    return body.get("message", {}).get("content", "")


def scripted_chat(replies: list[str]):
    """A chat_fn that returns canned replies, for tests and for offline demos."""
    queue = list(replies)

    def _fn(_messages: list[dict[str, str]]) -> str:
        if not queue:
            raise ModelError("scripted_chat ran out of replies")
        return queue.pop(0)

    return _fn


def available(host: str = DEFAULT_HOST, timeout: int = 5) -> bool:
    try:
        with urllib.request.urlopen(f"{host}/api/tags", timeout=timeout):
            return True
    except urllib.error.URLError:
        return False

In [ ]:
%%writefile src/explain.py
"""Explanation: turn a decision the engine has already made into words.

The model writes; it does not decide.  Everything factual in the output - which schemes,
which documents, which office, how much - is passed in from the engine and the YAML.
The model's job is to say it in the person's language, in an order that makes sense to
someone who is about to spend a day travelling to a counter.

Why this matters beyond politeness: the failure mode of a chat assistant here is a
fluent sentence that adds a scheme nobody checked, or softens "you need the death
certificate" into "you may want to bring some documents".  Both send a person on a
wasted trip.  So the sections are assembled in code and only the wording is generated,
and `verify_no_new_schemes` checks the output afterwards.
"""
from __future__ import annotations

from .engine import SchemeResult

SYSTEM_PROMPT = """You write short, plain messages for someone with limited literacy who
is about to travel to a government office.

You will be given a decision that has ALREADY been made. Your job is only to say it
clearly in the requested language.

RULES:
- Do NOT add any scheme that is not in the list you are given.
- Do NOT change any amount, document name, or office name.
- Do NOT say "you will definitely get" - say what they are entitled to apply for.
- Short sentences. No bullet symbols, no markdown, no English jargon in a Hindi message.
- Lead with what to do first."""


def render_brief(eligible: list[SchemeResult], uncertain: list[SchemeResult],
                 lang: str = "hi") -> str:
    """The factual brief handed to the model.  Everything here comes from the engine."""
    lines = []
    if eligible:
        lines.append("ENTITLED TO APPLY:")
        for r in eligible:
            s = r.scheme
            name = s["name_hi"] if lang == "hi" and "name_hi" in s else s["name_en"]
            lines.append(f"- {name} ({s['name_en']}): {s['benefit_en']}")
            lines.append(f"  where: {s.get('where', 'local office')}")
            lines.append(f"  documents: {'; '.join(s['documents'])}")
    if uncertain:
        lines.append("")
        lines.append("MAY ALSO QUALIFY, needs one more detail confirmed:")
        for r in uncertain:
            s = r.scheme
            name = s["name_hi"] if lang == "hi" and "name_hi" in s else s["name_en"]
            lines.append(f"- {name}: {s['benefit_en']}")
    if not eligible and not uncertain:
        lines.append("NOTHING MATCHED on the information given.")
    return "\n".join(lines)


def build_prompt(eligible: list[SchemeResult], uncertain: list[SchemeResult],
                 lang: str = "hi") -> list[dict[str, str]]:
    language = {"hi": "Hindi (Devanagari script)", "en": "English"}.get(lang, lang)
    brief = render_brief(eligible, uncertain, lang)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content":
            f"Write the message in {language}.\n\n{brief}\n\n"
            f"Write at most 120 words."},
    ]


def verify_no_new_schemes(text: str, allowed: list[SchemeResult],
                          all_schemes: dict) -> list[str]:
    """Check the generated message did not introduce a scheme nobody evaluated.

    A cheap, exact check: any scheme name in the rule file that appears in the message
    but is not in the allowed set is flagged.  It will not catch a paraphrase, and it is
    not meant to be the only safeguard - the factual content is assembled in code - but
    it does catch the specific failure of the model volunteering a scheme it happens to
    know about.
    """
    ok = set()
    for r in allowed:
        ok.add(r.scheme["name_en"].lower())
        if "name_hi" in r.scheme:
            ok.add(r.scheme["name_hi"])
    problems = []
    low = text.lower()
    for s in all_schemes["schemes"]:
        for key in (s["name_en"].lower(), s.get("name_hi", "")):
            if not key:
                continue
            present = key in (low if key == key.lower() else text)
            if present and key not in ok:
                problems.append(f"message mentions {s['name_en']}, which was not offered")
                break
    return problems


# Schemes name the same document differently - "Aadhaar card", "Aadhaar card of the
# applicant", "Aadhaar and bank passbook of the student". Grouping on the literal string
# produces a list telling someone to fetch their Aadhaar four times, which defeats the
# only purpose the checklist has. Each entry below maps to one physical thing a person
# has to obtain and carry.
DOC_CLASSES: list[tuple[str, tuple[str, ...]]] = [
    ("Aadhaar card", ("aadhaar",)),
    ("Bank account passbook", ("passbook", "bank account")),
    ("Ration card (BPL / Antyodaya)", ("ration card",)),
    ("Income certificate (from the Tehsil)", ("income certificate",)),
    ("Death certificate", ("death certificate",)),
    ("Caste certificate", ("caste certificate",)),
    ("MGNREGA job card", ("job card",)),
    ("Birth certificate of the child", ("birth certificate",)),
    ("School enrolment certificate", ("school enrolment", "enrolment certificate")),
    ("Passport photograph", ("photograph", "photo")),
    ("Land record (khatauni / khasra)", ("land record", "khatauni", "khasra")),
    ("Consent for Aadhaar seeding", ("seeding",)),
]


def normalise_document(raw: str) -> str:
    """Map a scheme's wording onto the physical document it refers to."""
    low = raw.lower()
    for label, keys in DOC_CLASSES:
        if any(k in low for k in keys):
            return label
    return raw


def document_checklist(eligible: list[SchemeResult]) -> list[tuple[str, list[str]]]:
    """Documents grouped so a person collects each one once, not once per scheme.

    A trip to a Tehsil for an income certificate can cost a day. The list is ordered by
    how many applications a document unlocks, so the most load-bearing errand is first.
    """
    need: dict[str, set[str]] = {}
    for r in eligible:
        for d in r.scheme["documents"]:
            need.setdefault(normalise_document(d), set()).add(r.scheme["name_en"])
    return sorted(((d, sorted(v)) for d, v in need.items()),
                  key=lambda kv: (-len(kv[1]), kv[0]))


def explain(eligible: list[SchemeResult], uncertain: list[SchemeResult],
            all_schemes: dict, chat_fn, lang: str = "hi") -> tuple[str, list[str]]:
    text = chat_fn(build_prompt(eligible, uncertain, lang))
    return text.strip(), verify_no_new_schemes(text, eligible + uncertain, all_schemes)

In [ ]:
%%writefile src/session.py
"""The loop: perceive, decide, ask the one question that matters, then explain.

The shape of the conversation is set by the engine, not the model.  After each answer
the engine recomputes what is still undecided and names the single fact whose value
would resolve the most benefit; that fact becomes the next question.  When no remaining
fact can change any outcome, the loop stops - which is usually after two or three
questions, not twenty.
"""
from __future__ import annotations

from dataclasses import dataclass, field

from .engine import evaluate, next_question, summarise
from .explain import explain
from .facts import QUESTIONS, Facts
from .perceive import perceive


@dataclass
class Turn:
    question: str | None
    utterance: str
    facts_after: Facts
    resolves: list[str] = field(default_factory=list)


@dataclass
class Session:
    schemes: dict
    chat_fn: object
    lang: str = "hi"
    facts: Facts = field(default_factory=Facts)
    turns: list[Turn] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)
    max_questions: int = 6

    def hear(self, utterance: str, question: str | None = None,
             expect_fact: str | None = None,
             resolves: list[str] | None = None) -> Facts:
        """Take one thing the person said and fold it into what we know."""
        new, warns, _raw = perceive(utterance, self.chat_fn, question, expect_fact)
        self.warnings.extend(warns)
        for name, value in new.known().items():
            # earlier answers win: a person correcting themselves says so explicitly,
            # and a later extraction inventing a value should not overwrite a stated one
            if getattr(self.facts, name) is None:
                self.facts = self.facts.with_(name, value)
        self.turns.append(Turn(question, utterance, self.facts, resolves or []))
        return self.facts

    def ask(self) -> tuple[str | None, str | None, list[str]]:
        """The next question to put to the person, or (None, ...) when done."""
        if len(self.turns) > self.max_questions:
            return None, None, []
        fact, diag = next_question(self.facts, self.schemes)
        if fact is None:
            return None, None, []
        text = QUESTIONS[fact][self.lang if self.lang in QUESTIONS[fact] else "en"]
        return fact, text, diag["resolves"].get(fact, [])

    def result(self):
        return summarise(evaluate(self.facts, self.schemes))

    def message(self) -> tuple[str, list[str]]:
        s = self.result()
        return explain(s["eligible"], s["uncertain"], self.schemes,
                       self.chat_fn, self.lang)

## 6 · The tests, and what needs no model to run

49 tests, no Ollama required. That is deliberate: everything that can deny someone a
pension is ordinary Python, tested like ordinary Python, and none of its behaviour depends
on what a language model felt like emitting today.

`tests/test_households.py` is the end-to-end one — eight complete households in, the
hand-derived list of schemes out. It also asserts that every *ineligible* scheme names the
rule that ruled it out, because an exclusion nobody can trace is an exclusion nobody can
appeal.

In [ ]:
%%writefile tests/test_households.py
"""End-to-end tests of the part that decides entitlement.

The other test files check components. These check the answer: eight complete households
in, the hand-derived list of schemes out. No model is involved, which is the point - the
component that can deny someone a pension is ordinary Python and is tested like it.
"""
from __future__ import annotations

import random

import pytest

from src.engine import evaluate, load_schemes, next_question, summarise
from src.facts import Facts
from tests.households import HOUSEHOLDS

import scripts_ask


@pytest.fixture(scope="module")
def schemes():
    return load_schemes("schemes/schemes.yaml")


@pytest.mark.parametrize("h", HOUSEHOLDS, ids=lambda h: h["id"])
def test_full_facts_reach_the_hand_derived_answer(h, schemes):
    got = summarise(evaluate(Facts(**h["facts"]), schemes))
    assert sorted(r.id for r in got["eligible"]) == sorted(h["expect_eligible"])
    assert not got["uncertain"], "a complete household should leave nothing uncertain"


@pytest.mark.parametrize("h", HOUSEHOLDS, ids=lambda h: h["id"])
def test_every_ruled_out_scheme_names_the_rule_that_ruled_it_out(h, schemes):
    """An exclusion nobody can trace is an exclusion nobody can appeal."""
    for res in evaluate(Facts(**h["facts"]), schemes):
        if res.status == "ineligible":
            assert res.failed, f"{res.id} is ineligible but names no failing rule"


@pytest.mark.parametrize("name", sorted(scripts_ask.STRATEGIES))
def test_question_order_never_changes_the_conclusion(name, schemes):
    """Asking in a different order may cost more questions. It must not change the answer.

    This is the property that makes it safe to optimise the interview at all: the order
    is a matter of the person's time, never of what they are entitled to.
    """
    pick = scripts_ask.STRATEGIES[name]
    for h in HOUSEHOLDS:
        r = scripts_ask.run(h, schemes, pick, random.Random(0))
        assert r["reached"] == sorted(h["expect_eligible"]), \
            f"{name} on {h['id']} reached {r['reached']}"


def test_interview_terminates_and_stops_asking_dead_questions(schemes):
    """next_question returns None once nothing further can move an outcome."""
    for h in HOUSEHOLDS:
        facts = Facts()
        for _ in range(len(h["facts"]) + 1):
            fact, _diag = next_question(facts, schemes)
            if fact is None:
                break
            facts = facts.with_(fact, h["facts"].get(fact))
        else:
            pytest.fail(f"{h['id']}: interview did not terminate")
        assert next_question(facts, schemes)[0] is None


def test_a_household_entitled_to_nothing_is_told_so(schemes):
    """The tool has to be able to say no, and say it without leaving anything uncertain."""
    h = next(x for x in HOUSEHOLDS if x["id"] == "urban_salaried")
    got = summarise(evaluate(Facts(**h["facts"]), schemes))
    assert got["eligible"] == []
    assert not got["uncertain"]
    assert len(got["ineligible"]) == len(schemes["schemes"])

In [ ]:
%%writefile tests/test_engine.py
"""Tests for the part that decides entitlement.

These run without a model. That is the point of the split: the component that can deny
someone a pension is ordinary Python with ordinary tests, and none of its behaviour
depends on what a language model felt like emitting today.
"""
from __future__ import annotations

import pytest
import yaml

from src.engine import (SchemeFileError, evaluate, load_schemes, next_question,
                        summarise)
from src.facts import Facts

SCHEMES = load_schemes("schemes/schemes.yaml")


# ---------------------------------------------------------------- the three-way split

def test_unknown_is_not_denial():
    """The failure this guards against is a tool that quietly says no.

    A widow who has stated nothing but her widowhood must come back as 'uncertain' for
    the widow pension, never 'ineligible'. Collapsing unknown into no is how entitlements
    go unclaimed.
    """
    res = {r.id: r for r in evaluate(Facts(applicant_is_widow=True), SCHEMES)}
    assert res["widow_pension"].status == "uncertain"
    assert "annual_income" in res["widow_pension"].missing


def test_definite_failure_is_ineligible():
    facts = Facts(applicant_is_widow=True, applicant_age=35, annual_income=90_000,
                  remarried=False)
    res = {r.id: r for r in evaluate(facts, SCHEMES)}
    assert res["widow_pension"].status == "ineligible"
    assert res["widow_pension"].failed[0].fact == "annual_income"


def test_all_rules_met_is_eligible():
    facts = Facts(applicant_is_widow=True, applicant_age=35, annual_income=40_000,
                  remarried=False)
    res = {r.id: r for r in evaluate(facts, SCHEMES)}
    assert res["widow_pension"].status == "eligible"


def test_false_and_unknown_differ():
    """`False` is an answer; `None` is a question still to ask."""
    unknown = evaluate(Facts(), SCHEMES)
    said_no = evaluate(Facts(owns_farmland=False), SCHEMES)
    by_id = lambda rs: {r.id: r.status for r in rs}
    assert by_id(unknown)["pm_kisan"] == "uncertain"
    assert by_id(said_no)["pm_kisan"] == "ineligible"


# ---------------------------------------------------------------- question selection

def test_question_prefers_the_fact_that_unlocks_most_money():
    """With nothing known, the BPL card gates the largest total benefit on the list."""
    fact, diag = next_question(Facts(), SCHEMES)
    assert fact == "bpl"
    assert set(diag["resolves"]["bpl"]) >= {"nfbs", "ayushman"}


def test_questions_stop_when_nothing_can_change():
    """Every scheme decided one way or the other means there is nothing left to ask."""
    facts = Facts(applicant_is_widow=False, applicant_age=30, annual_income=9_000_000,
                  bpl=False, residence="urban", house_type="pucca",
                  owns_motor_vehicle=True, has_girl_child=False,
                  has_school_age_child=False, owns_farmland=False,
                  primary_earner_died=False)
    fact, _ = next_question(facts, SCHEMES)
    assert fact is None


def test_answering_the_chosen_question_reduces_uncertainty():
    facts = Facts()
    before = len(summarise(evaluate(facts, SCHEMES))["uncertain"])
    for _ in range(6):
        fact, _ = next_question(facts, SCHEMES)
        if fact is None:
            break
        facts = facts.with_(fact, _plausible_value(fact))
    after = len(summarise(evaluate(facts, SCHEMES))["uncertain"])
    assert after < before


def _plausible_value(fact: str):
    return {"bpl": True, "applicant_age": 35, "annual_income": 40_000,
            "residence": "rural", "house_type": "kutcha"}.get(fact, False)


# ---------------------------------------------------------------- the rule file itself

def test_every_rule_references_a_declared_fact():
    """A typo in schemes.yaml must fail at load, not silently never match."""
    bad = {"schemes": [{"id": "x", "name_en": "X", "source": "s", "documents": [],
                        "rules": [{"fact": "aplicant_age", "op": "gte", "value": 60}]}]}
    with pytest.raises(SchemeFileError, match="unknown fact"):
        _validate(bad)


def test_every_fact_used_by_a_rule_is_askable():
    """If a rule depends on a fact, there has to be a way to ask a person about it."""
    from src.facts import QUESTIONS
    for s in SCHEMES["schemes"]:
        for r in s["rules"]:
            assert r["fact"] in QUESTIONS, f"{s['id']} needs a question for {r['fact']}"


def test_duplicate_scheme_ids_rejected():
    bad = {"schemes": [
        {"id": "x", "name_en": "X", "source": "s", "documents": [], "rules": []},
        {"id": "x", "name_en": "Y", "source": "s", "documents": [], "rules": []}]}
    with pytest.raises(SchemeFileError, match="duplicate"):
        _validate(bad)


def test_every_scheme_cites_a_source():
    for s in SCHEMES["schemes"]:
        assert s["source"].strip(), f"{s['id']} has no source document"


def _validate(obj, tmp=None):
    import tempfile, pathlib
    p = pathlib.Path(tempfile.mkstemp(suffix=".yaml")[1])
    p.write_text(yaml.safe_dump(obj))
    try:
        return load_schemes(p)
    finally:
        p.unlink()

In [ ]:
%%writefile tests/test_perceive.py
"""Tests for the boundary between the model and the rest of the system.

These use scripted model replies rather than a live model. They are not testing whether
Gemma extracts Hindi well - that is measured separately in reports/perception_eval.md.
They are testing that whatever the model returns, only well-formed facts get past this
layer, because everything downstream trusts what comes out of here.
"""
from __future__ import annotations

import json

from src.facts import Facts
from src.model import scripted_chat
from src.perceive import BIGHA_TO_HECTARE, coerce, perceive


def _run(payload) -> Facts:
    text = payload if isinstance(payload, str) else json.dumps(payload)
    facts, _warnings, _raw = perceive("...", scripted_chat([text]))
    return facts


def test_invented_field_is_dropped():
    """The model cannot introduce a fact no rule reads."""
    facts, warnings, _ = perceive(
        "...", scripted_chat([json.dumps({"bpl": True, "caste_category": "OBC"})]))
    assert facts.bpl is True
    assert not hasattr(facts, "caste_category")
    assert any("caste_category" in w for w in warnings)


def test_null_is_treated_as_unknown_not_false():
    facts = _run({"owns_farmland": None})
    assert facts.owns_farmland is None


def test_explicit_false_survives():
    facts = _run({"owns_farmland": False})
    assert facts.owns_farmland is False


def test_bighas_converted_in_code_not_by_the_model():
    facts = _run({"farmland_bighas": 2})
    assert facts.farmland_hectares == round(2 * BIGHA_TO_HECTARE, 3)
    assert facts.owns_farmland is True


def test_income_period_is_derived_consistently():
    assert _run({"annual_income": 60_000}).monthly_income == 5_000
    assert _run({"monthly_income": 5_000}).annual_income == 60_000


def test_wrong_type_is_dropped_with_a_warning():
    facts, warnings, _ = perceive(
        "...", scripted_chat([json.dumps({"applicant_age": "बहुत"})]))
    assert facts.applicant_age is None
    assert any("applicant_age" in w for w in warnings)


def test_unparseable_output_yields_no_facts_rather_than_wrong_ones():
    facts, warnings, _ = perceive("...", scripted_chat(["I think she qualifies!"]))
    assert facts.known() == {}
    assert warnings


def test_json_inside_a_code_fence_is_read():
    facts = _run('```json\n{"bpl": true}\n```')
    assert facts.bpl is True


def test_prose_around_the_json_is_tolerated():
    facts = _run('Here is the JSON:\n{"bpl": true}\nHope that helps.')
    assert facts.bpl is True


def test_question_context_reaches_the_model():
    """A bare 'yes' is meaningless without the question, so it has to be in the prompt."""
    seen: list = []

    def spy(messages):
        seen.append(messages)
        return "{}"

    perceive("हाँ", spy, question="क्या आपके पास बीपीएल कार्ड है?", expect_fact="bpl")
    last = seen[0][-1]["content"]
    assert "बीपीएल" in last
    assert "bpl" in last

In [ ]:
%%writefile tests/test_explain.py
"""Tests for the writing step.

The model writes the message but does not choose its content. These check that the
content it is handed is complete and that a message inventing a scheme is caught.
"""
from __future__ import annotations

from src.engine import evaluate, load_schemes, summarise
from src.explain import (build_prompt, document_checklist, render_brief,
                         verify_no_new_schemes)
from src.facts import Facts
from src.model import scripted_chat
from src.explain import explain

SCHEMES = load_schemes("schemes/schemes.yaml")


def _widow():
    facts = Facts(applicant_is_widow=True, applicant_age=35, annual_income=40_000,
                  remarried=False, bpl=True)
    return summarise(evaluate(facts, SCHEMES))


def test_brief_carries_amounts_documents_and_office():
    s = _widow()
    brief = render_brief(s["eligible"], s["uncertain"])
    assert "Widow Pension" in brief or "निराश्रित" in brief
    assert "1,000" in brief
    assert "Death certificate of the husband" in brief
    assert "sspy-up.gov.in" in brief


def test_prompt_forbids_adding_schemes():
    s = _widow()
    system = build_prompt(s["eligible"], s["uncertain"])[0]["content"]
    assert "not in the list" in system.lower()


def test_invented_scheme_in_the_message_is_flagged():
    s = _widow()
    bad = "आपको निराश्रित महिला पेंशन और पीएम किसान सम्मान निधि दोनों मिलेंगे।"
    problems = verify_no_new_schemes(bad, s["eligible"], SCHEMES)
    assert any("PM Kisan" in p for p in problems)


def test_a_faithful_message_is_not_flagged():
    s = _widow()
    good = "आप निराश्रित महिला पेंशन के लिए आवेदन कर सकती हैं।"
    assert verify_no_new_schemes(good, s["eligible"] + s["uncertain"], SCHEMES) == []


def test_documents_are_grouped_so_each_is_collected_once():
    s = _widow()
    checklist = document_checklist(s["eligible"])
    names = [d for d, _ in checklist]
    assert len(names) == len(set(names))
    # the document needed by the most schemes comes first, so one trip covers the most
    counts = [len(schemes) for _, schemes in checklist]
    assert counts == sorted(counts, reverse=True)


def test_explain_returns_the_models_words_and_any_problems():
    s = _widow()
    text, problems = explain(s["eligible"], s["uncertain"], SCHEMES,
                             scripted_chat(["आप पेंशन के लिए आवेदन कर सकती हैं।"]))
    assert "पेंशन" in text
    assert problems == []

In [ ]:
!python -m pytest tests -q

## 7 · How well the extraction actually works

The interview is measured above without a model. This section measures the one step that
genuinely depends on one: what Gemma writes into the form.

13 hand-labelled utterances — code-mixed speech, local units, regional number words,
answers that address a different question than the one asked, explicit denials, and two
things that must **not** become fields. Two scores, pulling opposite ways:

* **recall** — of what the person actually stated, how much was captured. A miss costs one
  extra question.
* **invention** — of what they did *not* state, how much was filled in anyway. An
  invention silently decides an entitlement on something nobody said.

In [ ]:
%%writefile tests/perception_cases.py
"""Hand-labelled utterances for measuring the perception step.

Written to cover what actually goes wrong at a counter rather than what is easy to
extract: code-mixed Hindi and English, local units, regional number words, answers that
address a different question than the one asked, and denials - which must come back as
`False`, not as silence.

`expect` lists only the fields the utterance genuinely determines. A field the speaker
did not establish is listed in `forbid`: emitting it is a hallucination even if the
guess would be reasonable, because downstream it becomes an entitlement decision nobody
asked about.
"""
from __future__ import annotations

CASES: list[dict] = [
    # ---- free narrative, Hindi
    dict(
        id="widow_narrative_hi",
        utterance="मेरे पति का पिछले साल देहांत हो गया। दो बच्चे स्कूल जाते हैं। "
                  "गाँव में रहते हैं, कच्चा घर है।",
        expect={"applicant_is_widow": True, "primary_earner_died": True,
                "years_since_earner_death": 1, "residence": "rural",
                "house_type": "kutcha", "has_school_age_child": True,
                "child_in_school": True},
        forbid=["annual_income", "bpl", "applicant_age"],
    ),
    dict(
        id="elderly_narrative_hi",
        utterance="मैं सत्तर साल का हूँ, अकेले रहता हूँ गाँव में। कोई कमाई नहीं है अब।",
        expect={"applicant_age": 70, "residence": "rural"},
        forbid=["applicant_is_widow", "bpl"],
    ),
    # ---- code-mixed, as people actually speak
    dict(
        id="hinglish_farmer",
        utterance="Do bigha zameen hai, gaon mein rehte hain. Ration card BPL wala hai.",
        expect={"owns_farmland": True, "farmland_hectares": 0.5,
                "residence": "rural", "bpl": True},
        forbid=["annual_income", "applicant_age"],
    ),
    dict(
        id="english_urban",
        utterance="I live in Prayagraj city, pucca house, I earn about 18000 a month.",
        expect={"residence": "urban", "house_type": "pucca", "monthly_income": 18000},
        forbid=["bpl", "owns_farmland"],
    ),
    # ---- regional number words
    dict(
        id="lakh_income",
        utterance="Saal bhar mein do lakh ke aas paas kama lete hain.",
        expect={"annual_income": 200000},
        forbid=["monthly_income_stated"],   # derived, not stated - see note below
    ),
    dict(
        id="hazaar_income_hi",
        utterance="महीने में बीस हज़ार मिल जाते हैं।",
        expect={"monthly_income": 20000},
        forbid=["annual_income_stated"],
    ),
    # ---- answers to a specific question
    dict(
        id="answer_yes_bpl",
        question="क्या आपके पास बीपीएल या अंत्योदय राशन कार्ड है?",
        expect_fact="bpl",
        utterance="हाँ, अंत्योदय कार्ड है।",
        expect={"bpl": True},
        forbid=[],
    ),
    dict(
        id="answer_no_vehicle",
        question="क्या घर में कोई मोटर वाहन है?",
        expect_fact="owns_motor_vehicle",
        utterance="नहीं, कोई गाड़ी नहीं है।",
        expect={"owns_motor_vehicle": False},
        forbid=[],
    ),
    dict(
        id="answer_no_remarriage",
        question="क्या आपने दोबारा विवाह किया है?",
        expect_fact="remarried",
        utterance="नहीं, दोबारा शादी नहीं की।",
        expect={"remarried": False},
        forbid=[],
    ),
    # ---- the answer addresses something else entirely
    dict(
        id="mismatched_answer",
        question="क्या घर में कोई मोटर वाहन है?",
        expect_fact="owns_motor_vehicle",
        utterance="पैंतीस साल।",
        expect={},
        forbid=["owns_motor_vehicle"],
        why="People answer the previous question, or the one they expected. Guessing "
            "here writes a fact nobody stated into an entitlement decision.",
    ),
    dict(
        id="answer_with_extra",
        question="आपकी उम्र क्या है?",
        expect_fact="applicant_age",
        utterance="अड़तालीस की हूँ, और दो बेटियाँ हैं।",
        expect={"applicant_age": 48, "has_girl_child": True, "girl_children_count": 2},
        forbid=[],
    ),
    # ---- things that must not become facts
    dict(
        id="hedged_income",
        utterance="Pata nahi kitna kamate hain, kabhi kuch kabhi kuch.",
        expect={},
        forbid=["annual_income", "monthly_income"],
        why="An explicit 'I don't know' is not a number. Filling one in here silently "
            "decides eligibility on a value the person never gave.",
    ),
    dict(
        id="bereavement_without_earner_claim",
        utterance="मेरी सास का देहांत हो गया था।",
        expect={},
        forbid=["applicant_is_widow", "primary_earner_died"],
        why="A death in the family is not the applicant's widowhood, and not "
            "necessarily the household's earner.",
    ),
]

# Derived fields are computed in code from a stated one, so a case that states a monthly
# figure will legitimately also carry the annual figure. `forbid` entries ending in
# `_stated` mark that distinction for the reader; the scorer ignores them.
DERIVED_SUFFIX = "_stated"

In [ ]:
%%writefile scripts_eval.py
"""Measure the perception step against hand-labelled utterances.

Reports two numbers that matter in opposite directions:

  recall     of the facts a person actually stated, how many were captured
  invention  of the facts they did not state, how many were filled in anyway

Invention is the one to watch. A missed fact costs a follow-up question; an invented one
silently decides an entitlement on something nobody said. The engine is built so silence
becomes a question rather than a denial, and that only holds if silence reaches it.
"""
from __future__ import annotations

import argparse
import sys
import time

sys.path.insert(0, ".")
from src.model import ollama_chat, available          # noqa: E402
from src.perceive import perceive                      # noqa: E402
from tests.perception_cases import CASES, DERIVED_SUFFIX  # noqa: E402


def close(a, b) -> bool:
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return abs(float(a) - float(b)) <= max(1.0, 0.02 * abs(float(b)))
    return a == b


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", type=int, default=1,
                    help="repeat each case, to see run-to-run stability")
    args = ap.parse_args()

    if not available():
        raise SystemExit("Ollama is not reachable; start it with `ollama serve`")

    hit = miss = invented = 0
    rows = []
    t0 = time.time()
    for case in CASES:
        got_all = []
        for _ in range(args.runs):
            facts, warnings, _raw = perceive(
                case["utterance"], ollama_chat,
                question=case.get("question"), expect_fact=case.get("expect_fact"))
            got_all.append(facts.known())
        got = got_all[0]

        wrong, missing, extra = [], [], []
        for k, v in case["expect"].items():
            if k in got and close(got[k], v):
                hit += 1
            elif k in got:
                wrong.append(f"{k}={got[k]!r} (wanted {v!r})")
                miss += 1
            else:
                missing.append(k)
                miss += 1
        for k in case["forbid"]:
            if k.endswith(DERIVED_SUFFIX):
                continue
            if k in got:
                extra.append(f"{k}={got[k]!r}")
                invented += 1

        stable = all(g == got_all[0] for g in got_all) if args.runs > 1 else None
        rows.append((case["id"], missing, wrong, extra, stable))

    total_expected = sum(len(c["expect"]) for c in CASES)
    total_forbidden = sum(len([k for k in c["forbid"]
                               if not k.endswith(DERIVED_SUFFIX)]) for c in CASES)
    print(f"\n{len(CASES)} cases, {args.runs} run(s), {time.time()-t0:.0f}s\n")
    for cid, missing, wrong, extra, stable in rows:
        flag = "ok  " if not (missing or wrong or extra) else "FAIL"
        note = []
        if missing:
            note.append("missed " + ",".join(missing))
        if wrong:
            note.append("wrong " + "; ".join(wrong))
        if extra:
            note.append("INVENTED " + "; ".join(extra))
        if stable is False:
            note.append("unstable across runs")
        print(f"  [{flag}] {cid:32s} {'  |  '.join(note)}")

    print(f"\nrecall    {hit}/{total_expected} = {hit/max(total_expected,1):.1%}"
          f"   (a miss costs one extra question)")
    print(f"invention {invented}/{total_forbidden} = {invented/max(total_forbidden,1):.1%}"
          f"   (an invention decides an entitlement nobody stated)")


if __name__ == "__main__":
    main()

In [ ]:
!python scripts_eval.py --runs 2

### The three failures this caught

The first run scored **79.2% recall / 21.4% invention**.

| the person said | the model wrote | why it matters |
|---|---|---|
| *"pata nahi kitna kamate hain, kabhi kuch kabhi kuch"* — I don't know, it varies | `annual_income: 0` | zero passes **every** income ceiling in the file. That one hallucination grants every income-tested scheme on the list and sends someone to a counter that will check. |
| *"मेरी सास का देहांत हो गया था"* — my mother-in-law died | `applicant_is_widow: true` | someone else's death recorded as the speaker's own widowhood, which requires a death certificate she cannot produce |

Both were fixed in the prompt with negative examples, and the zero was fixed **again** in
`coerce()`, which drops a stated income of zero with a warning. If a household genuinely
has no income, that is a BPL card and a zero-income certificate — not a number inferred
from a shrug.

The third failure appeared only *after* those fixes, which is the argument for keeping a
labelled set rather than eyeballing outputs: a flat *"नहीं"* to *have you remarried?* came
back as no field at all instead of `remarried: false`. The negative examples had
over-generalised into "anything touching marriage or bereavement is a trap".

**And that bug was only visible because the fields are tri-state.** Had unknown and False
been one value, it would have read `False`, the pension would have been granted, and the
interview would have moved on — right answer, broken mechanism, no way to notice. Fixed by
adding answer-shaped few-shots (`ANSWER_FEWSHOT`), which teach distrust of an unsupported
*inference* rather than distrust of a *topic*.

### The same code scores differently on different hardware

| | recall | invention |
|---|---:|---:|
| first run | 79.2% | 21.4% |
| after the prompt rules and the code backstop | 95.8% | 0% |
| after `ANSWER_FEWSHOT`, six local runs | **100%** | **0%** |
| **the cell above, on Kaggle's P100** | **95.8%** | **0%** |

Temperature is 0, so the last row is not sampling noise. It is the same weights on a
different backend — different kernels, different quantisation arithmetic, a different
order of operations. Temperature 0 buys determinism *within* a machine, not *across*
machines, and anyone quoting an extraction score without saying what it ran on is quoting
a number about their laptop.

What moved and what did not is the point:

- **Recall moved by one case**, and the cost is bounded and visible — the field stays
  unknown, so the engine asks about it, exactly as it would for any other unknown.
- **Invention stayed at 0% on both machines, on every run.** Not because the model behaved
  well today, but because it is enforced in `coerce()` and because entitlement was never
  the model's to decide.

A safety property that depends on a model scoring the same on somebody else's GPU is not a
safety property. The recall number is a cost estimate. The invention number is a
guarantee, and only because it does not depend on the model.

## 8 · One conversation, end to end

A widow in a village, starting from "I don't know what I'm entitled to".

The persona is a dictionary keyed by *field name*. The script does not know which questions
it will be asked or in what order — the engine picks each one from what is still undecided,
exactly as it would with a person in front of it. It opens with free narrative, which is
why it finishes in 8 questions rather than the 11.6 measured from a cold start: several
fields arrive in the first sentence.

Watch for the two things at the end that are not the list of schemes:

* **the document checklist, collapsed across schemes.** Six applications ask for Aadhaar
  under five different phrasings; the checklist says *Aadhaar card* once, ordered by how
  many applications each document unlocks — because a trip to the Tehsil for an income
  certificate can cost a day.
* **the audit trail**, down to the rule and the circular behind it.

In [ ]:
%%writefile demo.py
"""End-to-end walkthrough of one conversation.

Answers are matched to the questions the engine actually asks, which is the whole point:
the script does not know in advance what will be asked, because the engine decides that
from what is still undecided.
"""
from __future__ import annotations

import sys, time
sys.path.insert(0, ".")
from src.engine import audit_trail, load_schemes                     # noqa: E402
from src.explain import document_checklist                            # noqa: E402
from src.model import ollama_chat                                     # noqa: E402
from src.session import Session                                       # noqa: E402

# What this person would say if asked. A real deployment asks out loud; this stands in
# for the microphone so the walkthrough is reproducible.
PERSONA = {
    "bpl": "हाँ, अंत्योदय कार्ड है।",
    "applicant_age": "पैंतीस साल की हूँ।",
    "remarried": "नहीं, दोबारा शादी नहीं की।",
    "annual_income": "साल भर में मुश्किल से चालीस हज़ार।",
    "owns_farmland": "नहीं, अपनी ज़मीन नहीं है।",
    "owns_motor_vehicle": "नहीं, कोई गाड़ी नहीं है।",
    "monthly_income": "महीने में तीन-चार हज़ार।",
    "house_type": "कच्चा घर है।",
    "has_girl_child": "हाँ, दोनों बेटियाँ हैं।",
    "girl_children_count": "दो बेटियाँ।",
    "deceased_age_at_death": "चालीस के आसपास थे।",
    "years_since_earner_death": "पिछले साल की बात है।",
    "is_income_tax_payer": "नहीं, टैक्स नहीं भरते।",
    "holds_government_post": "नहीं, कोई सरकारी नौकरी नहीं।",
    "has_school_age_child": "हाँ, दोनों स्कूल जाती हैं।",
    "child_in_school": "हाँ, स्कूल जाती हैं।",
    "residence": "गाँव में रहते हैं।",
    "farmland_hectares": "ज़मीन नहीं है।",
    "primary_earner_died": "हाँ, वही कमाते थे।",
    "applicant_is_widow": "हाँ, विधवा हूँ।",
}

OPENING = ("मेरे पति का पिछले साल देहांत हो गया। दो बेटियाँ स्कूल जाती हैं। "
           "गाँव में रहते हैं, कच्चा घर है।")


def main() -> None:
    schemes = load_schemes("schemes/schemes.yaml")
    s = Session(schemes=schemes, chat_fn=ollama_chat, lang="hi", max_questions=8)
    t0 = time.time()

    print("SHE SAYS:\n  " + OPENING + "\n")
    s.hear(OPENING)
    print("  extracted:", s.facts.known(), "\n")

    while True:
        fact, question, resolves = s.ask()
        if fact is None:
            print("-- engine: no remaining question can change any outcome\n")
            break
        answer = PERSONA.get(fact, "पता नहीं।")
        print(f"ASK  [{fact}]  {question}")
        print(f"     (asked because it decides: {', '.join(resolves)})")
        print(f"SHE  {answer}")
        s.hear(answer, question=question, expect_fact=fact, resolves=resolves)
        print()

    r = s.result()
    n_q = len(s.turns) - 1
    print(f"=== after {n_q} questions, {time.time()-t0:.0f}s ===\n")
    print("ENTITLED TO APPLY:")
    for x in r["eligible"]:
        print(f"  - {x.scheme['name_en']}: {x.scheme['benefit_en']}")
    print("\nSTILL UNRESOLVED:")
    for x in r["uncertain"]:
        print(f"  - {x.scheme['name_en']} (needs {', '.join(x.missing)})")
    print("\nRULED OUT (with the rule that ruled it out):")
    for x in r["ineligible"]:
        print(f"  - {x.scheme['name_en']}: {x.failed[0].describe()}")

    print("\n=== DOCUMENTS TO COLLECT (each one once) ===")
    for doc, used_by in document_checklist(r["eligible"]):
        print(f"  {doc}  ->  {', '.join(used_by)}")

    print("\n=== AUDIT TRAIL for one decision ===")
    if r["eligible"]:
        print(audit_trail(r["eligible"][0]))

    print("\n=== MESSAGE TO THE PERSON (Gemma writes, engine decided) ===")
    text, problems = s.message()
    print(text)
    print("\nverification:", "no invented schemes" if not problems else problems)
    if s.warnings:
        print("perception warnings:", s.warnings)


if __name__ == "__main__":
    main()

In [ ]:
!python demo.py

## Limits

- **Nobody's entitlement is decided here.** The thresholds in this notebook were read off
  published scheme guidance so the engine would have something concrete to check. They are
  revised by notification and differ between states, and a programme officer would have to
  reconcile every line against the current circulars before this went near a real
  household.
- **8 schemes, 8 households, 13 utterances.** Enough to catch a systematic failure and a
  regression; not a confidence interval.
- **Hindi and English are measured; other languages are not.** Gemma supports more. A claim
  with no labelled set behind it is not a claim.
- **Written input, not spoken.** Real deployment goes through ASR, which adds its own
  errors upstream of everything measured here.
- **`verify_no_new_schemes` is exact-match.** It catches the model naming a scheme nobody
  evaluated and would miss a paraphrase. It is a backstop; the factual content is assembled
  in code.
- **Nothing is filed.** The tool says what to bring and where to go.

## What I would build next

1. **Voice, both directions.** The person this is for may not read; the worker beside them
   does. ASR plus TTS closes that gap and changes nothing above it.
2. **A correction loop for caseworkers.** When a decision is wrong, the operator should be
   able to record that *against the rule that produced it* — possible only because a rule
   produced it. That is the dataset that improves a programme, and a model deciding
   end-to-end cannot generate it.
3. **Per-district rule files.** Eight schemes were enough to build and measure against; a district's
   real list is a data-collection problem, not a modelling one.

---

**Repository, with the full engineering log:** https://github.com/yunyiliu/kaggle-gemma-yojana-sarathi